# **Objective**:

- Use TF-IDF with different feature limits to compare performance.
- Train and evaluate a Logistic Regression model
- Handle class imbalance using class weights.

##Import Necessary Libraries

In [ ]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
We will use the GPU: Tesla T4


In [ ]:
pip install livelossplot

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, accuracy_score, classification_report
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
import joblib

In [ ]:
!pip install --upgrade tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 89.5 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.18.0
    Uninstalling tensorflow-2.18.0:
      Successfully uninstalled tensorflow-2.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from keras import Model, Input
from keras.layers import LSTM, Embedding, Dense
from keras.layers import TimeDistributed, SpatialDropout1D, Bidirectional
from keras.callbacks import ModelCheckpoint, EarlyStopping
from livelossplot.tf_keras import PlotLossesCallback

##Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_cleaned = pd.read_csv("/content/drive/MyDrive/Arabic_Dialect_Classification/cleaned_dataset.csv", engine="python")


In [ ]:
df_cleaned

,Cleaned_text,SpeakerDialect
0,وضح كلامك يا مغيث,Najdi
1,تري راسي ما عاد يتحمل الغازك,Najdi
2,سلامه راسك يا ابو مسامح,Najdi
3,ما يصير يا ابو مسامح تخلي البنت في البيت دون ...,Najdi
4,ويش فيها لقعدت في البيت دون امها ما هو ده بيت...,Najdi
...,...,...
241828,لازم تشيلها عشان تصير سعيد في حياتك,Najdi
241829,انا سويت كل هذا علي شانك لا واله مريم مريم عط...,More than 1 speaker اكثر من متحدث
241830,مريم,Najdi
241831,لا لا,Najdi


In [ ]:
pd.set_option('display.max_rows',500)
pd.set_option('display.max_colwidth',500)

In [ ]:
df_cleaned.isnull().values.any()


np.False_

In [ ]:
df_cleaned.isnull().sum()

,0
Cleaned_text,0
SpeakerDialect,0


##Data Splitting

In [ ]:
x=df_cleaned.drop(['SpeakerDialect'], axis=1)
y=df_cleaned['SpeakerDialect']

In [ ]:
print("shape of x :",x.shape)
print("shape of y  :" ,y.shape)

shape of x : (241833, 1)
shape of y  : (241833,)


In [ ]:
x_train ,x_test ,y_train , y_test = train_test_split(x,y,test_size=0.15, random_state=24, shuffle=True,stratify=y)

In [ ]:
x_train, x_val , y_train, y_val = train_test_split(x_train, y_train, test_size=0.10, random_state=42,stratify=y_train)

In [ ]:
x_train.isnull().sum()

,0
Cleaned_text,0


# Tokenizer and Model

## ML Models

We extract all **unique words** from the `Cleaned_text` column of the training set using a Python `set`. This helps us:

- Get an understanding of the vocabulary size.
- Prepare for building the tokenizer or embedding layer.

Steps:
1. Iterate over each cleaned text sample.
2. Tokenize by splitting on whitespace.
3. Add tokens to a set, which automatically filters duplicates.

The final count gives us the **number of distinct words** present in the training data.


In [ ]:
# Use a Set to Store Unique Words
unique_words = set()

for text in x_train['Cleaned_text']:
    words = text.split()  # Tokenize
    unique_words.update(words)

# Number of Unique Words
num_unique_words = len(unique_words)
print(f'Number of unique words: {num_unique_words}')

Number of unique words: 135637


**Term Frequency-Inverse Document Frequency (TF-IDF):**
-  TF-IDF assigns weights to terms based on their frequency in individual documents and their inverse frequency across all documents. This technique helps capture the importance of terms in distinguishing between documents.

In [ ]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
x_train_tfidf = vectorizer.fit_transform(x_train['Cleaned_text'])
x_val_tfidf = vectorizer.transform(x_val['Cleaned_text'])
x_test_tfidf = vectorizer.transform(x_test['Cleaned_text'])

In [ ]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
# Check the mapping
print("Label Mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

Label Mapping: {'Egyptian': np.int64(0), 'Hijazi': np.int64(1), 'Iraqi': np.int64(2), 'Janubi': np.int64(3), 'Khaliji': np.int64(4), 'Levantine': np.int64(5), 'Maghrebi': np.int64(6), 'ModernStandardArabic': np.int64(7), 'More than 1 speaker اكثر من متحدث': np.int64(8), 'Najdi': np.int64(9), 'Notapplicable': np.int64(10), 'Shamali': np.int64(11), 'Unknown': np.int64(12), 'Yemeni': np.int64(13)}


##Logistic regression

In [ ]:
# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(x_train_tfidf, y_train_encoded)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
# Evaluate on validation set
y_val_pred = model.predict(x_val_tfidf)
val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

Validation Accuracy: 0.5198


In [ ]:
# Evaluate on test set
y_test_pred = model.predict(x_test_tfidf)
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Accuracy: 0.5232


In [ ]:
from sklearn.metrics import classification_report

# Get full label range
labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# Print classification report
print(classification_report(
    y_test_encoded,
    y_test_pred,
    labels=labels,
    target_names=target_names,
    zero_division=0  # Avoids division-by-zero warnings
))


                                   precision    recall  f1-score   support

                         Egyptian       0.50      0.05      0.09       305
                           Hijazi       0.43      0.29      0.35      5225
                            Iraqi       0.00      0.00      0.00         0
                           Janubi       0.00      0.00      0.00        15
                          Khaliji       0.34      0.11      0.17      4274
                        Levantine       0.50      0.01      0.01       142
                         Maghrebi       0.00      0.00      0.00         6
             ModernStandardArabic       0.50      0.18      0.27       614
More than 1 speaker اكثر من متحدث       0.64      0.66      0.65      7520
                            Najdi       0.51      0.81      0.63     13599
                    Notapplicable       0.00      0.00      0.00        58
                          Shamali       0.00      0.00      0.00        19
                        

### use 10000  Unique Words


In [ ]:
# TF-IDF Vectorization
vectorizer_ = TfidfVectorizer(max_features=10000)
x_train_tfidf = vectorizer_.fit_transform(x_train['Cleaned_text'])
x_val_tfidf = vectorizer_.transform(x_val['Cleaned_text'])
x_test_tfidf = vectorizer_.transform(x_test['Cleaned_text'])

In [ ]:
# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(x_train_tfidf, y_train_encoded)


LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
# Evaluate on validation set
y_val_pred = model.predict(x_val_tfidf)
val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
val_Precision = precision_score(y_val_encoded, y_val_pred, average='macro')
print(f'Validation Accuracy: {val_accuracy:.4f}')
print(f'Precision: {val_Precision:.4f}')

Validation Accuracy: 0.5218
Precision: 0.2651


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Evaluate on test set
y_test_pred = model.predict(x_test_tfidf)
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Accuracy: 0.5262


In [ ]:
from sklearn.metrics import classification_report

# Get full label range
labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# Print classification report
print(classification_report(
    y_test_encoded,
    y_test_pred,
    labels=labels,
    target_names=target_names,
    zero_division=0  # Avoids division-by-zero warnings
))


                                   precision    recall  f1-score   support

                         Egyptian       0.57      0.05      0.10       305
                           Hijazi       0.43      0.31      0.36      5225
                            Iraqi       0.00      0.00      0.00         0
                           Janubi       0.00      0.00      0.00        15
                          Khaliji       0.34      0.12      0.18      4274
                        Levantine       0.50      0.03      0.05       142
                         Maghrebi       0.00      0.00      0.00         6
             ModernStandardArabic       0.53      0.18      0.27       614
More than 1 speaker اكثر من متحدث       0.63      0.65      0.64      7520
                            Najdi       0.52      0.81      0.63     13599
                    Notapplicable       0.00      0.00      0.00        58
                          Shamali       0.00      0.00      0.00        19
                        

Use All Unique Words

In [ ]:
# TF-IDF Vectorization
vectorizer_ = TfidfVectorizer(max_features=num_unique_words)
x_train_tfidf = vectorizer_.fit_transform(x_train['Cleaned_text'])
x_val_tfidf = vectorizer_.transform(x_val['Cleaned_text'])
x_test_tfidf = vectorizer_.transform(x_test['Cleaned_text'])

In [ ]:
# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(x_train_tfidf, y_train_encoded)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
# Evaluate on validation set
y_val_pred = model.predict(x_val_tfidf)
val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
val_Precision = precision_score(y_val_encoded, y_val_pred, average='macro')
print(f'Validation Accuracy: {val_accuracy:.4f}')
print(f'Precision: {val_Precision:.4f}')

Validation Accuracy: 0.5208
Precision: 0.2567


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Evaluate on test set
y_test_pred = model.predict(x_test_tfidf)
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Accuracy: 0.5271


In [ ]:
from sklearn.metrics import classification_report

# Get full label range
labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# Print classification report
print(classification_report(
    y_test_encoded,
    y_test_pred,
    labels=labels,
    target_names=target_names,
    zero_division=0  # Avoids division-by-zero warnings
))


                                   precision    recall  f1-score   support

                         Egyptian       0.46      0.04      0.07       305
                           Hijazi       0.45      0.32      0.37      5225
                            Iraqi       0.00      0.00      0.00         0
                           Janubi       0.00      0.00      0.00        15
                          Khaliji       0.36      0.14      0.20      4274
                        Levantine       0.33      0.02      0.04       142
                         Maghrebi       0.00      0.00      0.00         6
             ModernStandardArabic       0.57      0.18      0.28       614
More than 1 speaker اكثر من متحدث       0.60      0.64      0.62      7520
                            Najdi       0.53      0.80      0.64     13599
                    Notapplicable       0.00      0.00      0.00        58
                          Shamali       0.00      0.00      0.00        19
                        

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get unique classes
classes = np.unique(y_train_encoded)

# Compute weights
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_encoded)

# Make dict: {class_index: weight}
class_weight_dict = dict(zip(classes, weights))

# Train model
model = LogisticRegression(class_weight=class_weight_dict, max_iter=1000, random_state=42)
model.fit(x_train_tfidf, y_train_encoded)


LogisticRegression(class_weight={np.int64(0): np.float64(8.508968816116273),
                                 np.int64(1): np.float64(0.4959068026955594),
                                 np.int64(2): np.float64(6607.214285714285),
                                 np.int64(3): np.float64(167.27124773960216),
                                 np.int64(4): np.float64(0.6063055091272572),
                                 np.int64(5): np.float64(18.22679802955665),
                                 np.int64(6): np.float64(412.95089285714283),
                                 np.int64(7): np.float64(4.223211432223896),
                                 np.int64(8): np.float64(0.34455643959711546),
                                 np.int64(9): np.float64(0.19054142016709788),
                                 np.int64(10): np.float64(44.34372003835091),
                                 np.int64(11): np.float64(134.8411078717201),
                                 np.int64(12): np.float64(0.5833669685426704),
                                 np.int64(13): np.float64(45.88343253968254)},
                   max_iter=1000, random_state=42)

In [ ]:
# Evaluate on test set
y_test_pred = model.predict(x_test_tfidf)
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Accuracy: 0.4266


In [ ]:
from sklearn.metrics import classification_report

# Get full label range
labels = list(range(len(label_encoder.classes_)))
target_names = label_encoder.classes_

# Print classification report
print(classification_report(
    y_test_encoded,
    y_test_pred,
    labels=labels,
    target_names=target_names,
    zero_division=0  # Avoids division-by-zero warnings
))


                                   precision    recall  f1-score   support

                         Egyptian       0.08      0.39      0.13       305
                           Hijazi       0.38      0.37      0.38      5225
                            Iraqi       0.00      0.00      0.00         0
                           Janubi       0.01      0.13      0.01        15
                          Khaliji       0.27      0.36      0.31      4274
                        Levantine       0.06      0.37      0.10       142
                         Maghrebi       0.03      0.17      0.05         6
             ModernStandardArabic       0.20      0.63      0.30       614
More than 1 speaker اكثر من متحدث       0.60      0.61      0.60      7520
                            Najdi       0.73      0.40      0.51     13599
                    Notapplicable       0.04      0.47      0.07        58
                          Shamali       0.01      0.16      0.02        19
                        